# Decision relevance audit

## tl;dr

You have no demonstrated allocation-policy benefit. On matching pre-IPO cases, you get AUC 0.705 from negative log offer price and 0.702 from logistic regression. You get day-20 AUC 0.727 from realized volatility. The 69% pre-IPO terminal-gain figure reproduces as a mixed-price-basis proxy; it does not establish allocator P&L.

## Context & Methods

You audit the existing binary drawdown forecasts. You do not fit a new model or inspect prospective outcomes. You use one-feature directions from the review, so you should treat the ranking check as post-hoc.

### Key Assumptions

You pair model and feature scores on matching held-out issuer IDs. You exclude missing offer prices from both sides of the pre-IPO ranking comparison. You measure terminal ratios with the vendor-adjusted prices stored in the snapshot. You need original-share accounting before interpreting a nominal-offer ratio as an investment return.

In [1]:
from pathlib import Path
import json
import runpy

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "research/expanded/results.json.gz").exists())
audit = runpy.run_path(str(root / "research/decision/audit.py"))["audit"]
result = audit()

## Data

You read the committed compressed input, dataset, and forecast snapshots. You can inspect their decompressed SHA-256 hashes below. You need the locked challenger dependencies to rerun the audit script; notebook execution also needs nbformat, nbclient, and ipykernel.

In [2]:
print(json.dumps(result["inputs"], indent=2))

{
  "dataset.json.gz": "f0d2aecc8e58cbb3fe1c49a8a4141ec7801650b69f3f44c071f65588026c748d",
  "results.json.gz": "2c228d0b3760177617fd1854f904542a629139bc4f4a3a67a2cad5fda9a0916e",
  "input.json.gz": "34c96a2672c21879e36976b07415befa614c79e352d4638d6784fd595e0e9669"
}


## Results

You compare each one-feature score with model scores on the same cases. You read the terminal-gain counts among drawdown events with an available return proxy.

In [3]:
for stage, metrics in result["stages"].items():
    print("Stage", stage)
    print(json.dumps({key: value for key, value in metrics.items() if key != "yearly"}, indent=2))

Stage 0
{
  "held_out": 980,
  "single_feature": "log_offer_price",
  "direction": -1,
  "ranking_cases": 953,
  "ranking_missing": 27,
  "single_feature_auc": 0.7049613538504977,
  "model_aucs_same_cases": {
    "logistic": 0.7019040728453448,
    "boosted_trees": 0.6926329674595892,
    "catboost": 0.6707246594197783,
    "tabnet": 0.6549772358297452
  },
  "return_proxy_cases": 953,
  "event_cases_with_return_proxy": 496,
  "events_with_nonnegative_terminal_proxy": 342,
  "share_events_with_nonnegative_terminal_proxy": 0.6895161290322581,
  "events_never_below_entry_proxy": 310,
  "yearly_coverage_event_correlation": 0.8102678934158548,
  "median_first_close_offer_proxy": 0.27045458013361157,
  "first_close_offer_proxy_event_correlation": 0.06360341048182919
}
Stage 20
{
  "held_out": 964,
  "single_feature": "daily_volatility",
  "direction": 1,
  "ranking_cases": 964,
  "ranking_missing": 0,
  "single_feature_auc": 0.7269551742645913,
  "model_aucs_same_cases": {
    "logistic": 0

### Coverage association

You compare price-match coverage with event rates across sixteen IPO years. These associations cannot distinguish missing-history bias from changes in issuer mix or market conditions.

In [4]:
for stage, metrics in result["stages"].items():
    print("Stage", stage, "year | eligible | coverage | event rate")
    for row in metrics["yearly"]:
        print(f"{row['year']} | {row['eligible']} | {row['coverage']:.3f} | {row['event_rate']:.3f}")

Stage 0 year | eligible | coverage | event rate
2010 | 44 | 0.266 | 0.136
2011 | 23 | 0.170 | 0.217
2012 | 37 | 0.285 | 0.054
2013 | 63 | 0.286 | 0.079
2014 | 75 | 0.273 | 0.253
2015 | 40 | 0.263 | 0.275
2016 | 41 | 0.438 | 0.220
2017 | 55 | 0.350 | 0.145
2018 | 103 | 0.536 | 0.272
2019 | 82 | 0.521 | 0.329
2020 | 124 | 0.506 | 0.508
2021 | 221 | 0.487 | 0.516
2022 | 64 | 0.695 | 0.812
2023 | 75 | 0.653 | 0.533
2024 | 135 | 0.831 | 0.519
2025 | 176 | 0.842 | 0.619
Stage 20 year | eligible | coverage | event rate
2010 | 44 | 0.266 | 0.136
2011 | 23 | 0.170 | 0.087
2012 | 36 | 0.285 | 0.083
2013 | 62 | 0.286 | 0.097
2014 | 75 | 0.273 | 0.253
2015 | 40 | 0.263 | 0.250
2016 | 41 | 0.438 | 0.195
2017 | 55 | 0.350 | 0.182
2018 | 101 | 0.536 | 0.337
2019 | 81 | 0.521 | 0.346
2020 | 123 | 0.506 | 0.358
2021 | 219 | 0.487 | 0.484
2022 | 62 | 0.695 | 0.597
2023 | 74 | 0.653 | 0.527
2024 | 134 | 0.831 | 0.537
2025 | 173 | 0.842 | 0.509


## Takeaways

You need a target tied to the actor's entry price and an executable reference policy. You should evaluate the allocator design only after you reconcile share bases and preserve delisting outcomes. You need dated filing inputs before you test price revision and deal structure. You cannot use a later 424B4 as evidence of earlier availability without another dated source.

You can inspect the draft decision contract in `research/decision/protocol.json`. You have no fitted allocator model or completed execution backtest.